# LOT 2026 - Cognitive modeling meets computational linguistics (Yevgen Matusevych)
## Day 4 practical
# Crosslingual structural priming in bilingual language models

**Structural priming** is a classic finding in psycholinguistics: after hearing or reading a
sentence with a particular structure, people are more likely to reuse that structure - even with
different words. In bilinguals, priming can carry *across languages* (a Dutch sentence priming an
English one), which is taken as evidence that the two languages share grammatical representations.

In this practical you measure structural priming in small **bilingual GPT-2 models** (the "B-GPT"
models of Arnett et al., 2025), using the Dutch-English sentence materials of Schoonbaert et al.
(2007). You will:

1. Load a trained bilingual model.
2. Measure whether a prime sentence makes a same-structured target sentence more probable.
3. See how the effect changes when you vary the **direction** of priming, the **amount of
   second-language training**, and whether prime and target share a **verb**.

A free Colab CPU runtime is enough - the models are small (~0.1B parameters).

## 1. Setup

Run this once. We install `minicons` (for the probability calculations) and ask `pip` for a
`transformers`/`tokenizers` version that can read these models' tokenizers.

In [ ]:
%pip install -q minicons sentencepiece "transformers<5" "tokenizers<0.21" "huggingface_hub<1.0"

## 2. (Optional) Hugging Face access token

The models are public, so you normally need nothing here. Only set a token if you hit rate-limit
errors - create one at <https://huggingface.co/settings/tokens> and paste it below.

In [ ]:
HF_TOKEN = ""   # leave empty unless you are rate-limited

## 3. Load a bilingual model

The models are named `catherinearnett/B-GPT_<L1>_<L2>_<schedule>`. For example
`B-GPT_en_nl_simultaneous` saw **English first (L1)** and then a mix of English and Dutch
(**Dutch is its L2**), while `B-GPT_nl_en_simultaneous` saw **Dutch first**.

**Convention (following Arnett et al.):** throughout this notebook we always **prime in the model's
first language (L1) and measure the effect on its second language (L2)** - the L1→L2 direction. Our
main model is `model_nl_en`, which learned **Dutch first (L1)** and **English second (L2)**, so we
prime in Dutch and target English (this keeps the *target* language English, the direction Arnett et
al. find most robust). To change the target language you load the model with the opposite exposure
order (Section 7), rather than swapping prime and target on one model.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from minicons import scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_scorer(model_id, revision=None):
    """Download a bilingual model and return an object that can score sentences.

    Parameters
    ----------
    model_id : str
        A Hugging Face repository, e.g. "catherinearnett/B-GPT_en_nl_simultaneous".
    revision : str, optional
        A specific training checkpoint (a branch or tag name). If omitted, the
        fully trained model is used.

    Returns
    -------
    minicons.scorer.IncrementalLMScorer
        Call its ``.conditional_score(...)`` method to get log-probabilities.
    """
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, token=token)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, token=token)
    return scorer.IncrementalLMScorer(model, device=DEVICE, tokenizer=tokenizer)


# model_nl_en: Dutch is L1, English is L2. We prime in Dutch (L1), target English (L2).
model_nl_en = load_scorer("catherinearnett/B-GPT_nl_en_simultaneous")
print("Model loaded on", DEVICE)

## 4. Load the stimuli

We use the Dutch-English priming sentences from Schoonbaert et al. (2007), available from the
project's OSF page (<https://osf.io/5cw2e/>). **Download the TSV and upload it to Colab** (the Files
panel on the left → upload button); it will appear next to this notebook.

Each row is one *prime → target* pairing, with columns telling us the language and structure of each
sentence. The two structures are the dative alternation:

- **PO** (prepositional object): *the waiter gives a book **to** the doctor*
- **DO** (double object): *the waiter gives the doctor a book*

If the file is not present, the cell below falls back to a tiny built-in demo set so everything
still runs - but the demo set is only a placeholder, not real data.

In [ ]:
import os
import pandas as pd

STIMULI_PATH = "schoonbaert_2007.tsv"   # the file you upload to Colab

# Columns expected in the stimulus file.
STIMULUS_COLUMNS = ["Set", "ExtraCondition", "PrimeLanguage", "PrimeType",
                    "TargetType", "TargetLanguage", "PrimeSentence", "TargetSentence"]


def build_demo_stimuli():
    """Build a tiny 3-item demo set in the same format as the real file.

    Used only when the real stimulus file has not been uploaded, so that the
    rest of the notebook can run. Not suitable for drawing conclusions.
    """
    items = [
        {"prime":  {"en_po": "The monk throws a hat to the sailor",
                    "en_do": "The monk throws the sailor a hat",
                    "nl_po": "De monnik gooit een hoed aan de matroos",
                    "nl_do": "De monnik gooit de matroos een hoed"},
         "target": {"en_po": "The policeman throws a gun to the clown",
                    "en_do": "The policeman throws the clown a gun",
                    "nl_po": "De agent gooit een geweer aan de clown",
                    "nl_do": "De agent gooit de clown een geweer"}},
        {"prime":  {"en_po": "The waiter gives a book to the doctor",
                    "en_do": "The waiter gives the doctor a book",
                    "nl_po": "De ober geeft een boek aan de dokter",
                    "nl_do": "De ober geeft de dokter een boek"},
         "target": {"en_po": "The nurse sends a letter to her sister",
                    "en_do": "The nurse sends her sister a letter",
                    "nl_po": "De verpleegster stuurt een brief aan haar zus",
                    "nl_do": "De verpleegster stuurt haar zus een brief"}},
        {"prime":  {"en_po": "The man sells a bike to the student",
                    "en_do": "The man sells the student a bike",
                    "nl_po": "De man verkoopt een fiets aan de student",
                    "nl_do": "De man verkoopt de student een fiets"},
         "target": {"en_po": "The guide shows a map to the tourists",
                    "en_do": "The guide shows the tourists a map",
                    "nl_po": "De gids toont een kaart aan de toeristen",
                    "nl_do": "De gids toont de toeristen een kaart"}},
    ]
    code_to_name = {"en": "English", "nl": "Dutch"}
    rows = []
    for set_number, item in enumerate(items, start=1):
        # Cross every prime (language x structure) with every target (language x structure).
        for prime_lang in ("en", "nl"):
            for prime_struct in ("po", "do"):
                for target_lang in ("en", "nl"):
                    for target_struct in ("po", "do"):
                        rows.append({
                            "Set": f"Set{set_number}",
                            "ExtraCondition": "same verb",
                            "PrimeLanguage":  f"{code_to_name[prime_lang]}_Prime",
                            "PrimeType":      f"{prime_struct.upper()}_Prime",
                            "TargetType":     f"{target_struct.upper()}_Target",
                            "TargetLanguage": f"{code_to_name[target_lang]}_Target",
                            "PrimeSentence":  item["prime"][f"{prime_lang}_{prime_struct}"],
                            "TargetSentence": item["target"][f"{target_lang}_{target_struct}"],
                        })
    return pd.DataFrame(rows)


def load_stimuli(path=STIMULI_PATH):
    """Load the real stimulus file if it is present, otherwise the demo set."""
    if os.path.exists(path):
        stimuli = pd.read_csv(path, sep="\t")
        # Arnett et al. exclude Set16 from the Schoonbaert materials; do the same so
        # our numbers line up with theirs.
        stimuli = stimuli[stimuli["Set"] != "Set16"].reset_index(drop=True)
        print(f"Loaded {len(stimuli)} rows from '{path}' (Set16 excluded, as in Arnett et al.).")
    else:
        stimuli = build_demo_stimuli()
        print(f"'{path}' not found - using the built-in demo set ({len(stimuli)} rows). "
              f"Upload the real file for meaningful results.")
    return stimuli


stimuli = load_stimuli()
stimuli.head()

## 5. Measuring structural priming

We measure priming the way Arnett et al. (2025) do - as a difference in *normalized* probabilities.

For a given prime, we ask how much of the model's probability for the two competing target structures
goes to one of them. This is the **relative probability** of, say, the PO target:

$$P_N(\text{PO target}\mid\text{prime}) \;=\; \frac{P(\text{PO target}\mid\text{prime})}{P(\text{PO target}\mid\text{prime}) + P(\text{DO target}\mid\text{prime})}$$

a number in $[0, 1]$ giving the model's preference for the PO structure after that prime. Structural
priming is then the change in that preference when the prime is **congruent** (same structure as the
target) versus **incongruent** (the other structure):

$$\text{priming effect} \;=\; P_N(\text{target}\mid\text{congruent prime}) \;-\; P_N(\text{target}\mid\text{incongruent prime})$$

A **positive** value means a congruent prime shifts the model toward the matching structure; across
languages, that is evidence the structure is represented in a way *shared* between the two languages.
Normalizing this way puts the effect on a bounded scale comparable to the human measure (the
proportion of times a speaker reuses a structure), and divides out a prime's general tendency to make
everything more or less probable. Each item therefore needs all four prime x target combinations
(PO/DO prime x PO/DO target). We get the probabilities from `minicons`; prime and target are joined
with ". ".

In [ ]:
import math


def conditional_logprob(model, prime, target):
    """Return log P(target | prime): how probable the target is after the prime.

    The prime and target are joined with ". " (as in the original materials).
    minicons subtracts the prime's own probability, so this is a true conditional.
    A higher (less negative) value means the target is more expected.
    """
    sum_over_tokens = lambda token_logprobs: token_logprobs.sum(0).item()
    return model.conditional_score([prime], [target],
                                   separator=". ", reduction=sum_over_tokens)[0]


def relative_probability(logp_target, logp_alternative):
    """Normalized probability of a target against its structural alternative.

    Given log P(target | prime) and log P(other-structure target | prime), return
    P(target) / (P(target) + P(other)) in [0, 1]. Implemented as a numerically
    stable logistic of the log-probability difference, so the very small sentence
    probabilities do not underflow.
    """
    difference = logp_target - logp_alternative
    if difference >= 0:
        return 1.0 / (1.0 + math.exp(-difference))
    exp_diff = math.exp(difference)
    return exp_diff / (1.0 + exp_diff)


def structure_of(label):
    """Extract the structure from a label: 'PO_Prime' -> 'PO', 'DO_Target' -> 'DO'."""
    return label.split("_")[0]


# Map the short language codes we use to the names used in the stimulus file.
LANGUAGE_NAME = {"en": "English", "nl": "Dutch", "pl": "Polish", "el": "Greek"}


def run_priming_experiment(model, stimuli, prime_lang, target_lang, verb_condition=None):
    """Compute the priming effect for every item in one direction.

    For each item we score all four prime x target combinations, turn each pair of
    target probabilities into a relative probability, and define the priming effect
    as the target's relative probability after a congruent prime minus after an
    incongruent one.

    Parameters
    ----------
    model : IncrementalLMScorer
        A model loaded with ``load_scorer``.
    stimuli : pandas.DataFrame
        The stimulus table.
    prime_lang, target_lang : str
        Short language codes, e.g. "nl" (prime) and "en" (target).
    verb_condition : str, optional
        Restrict to "same verb" or "different verb" primes; None uses both.

    Returns
    -------
    pandas.DataFrame
        Two rows per item (one per target structure), with the relative probability
        after a congruent prime ("prob_match"), after an incongruent prime
        ("prob_mismatch"), and the priming effect ("PE" = their difference).
    """
    # Keep only rows for this prime-language -> target-language direction.
    rows = stimuli[
        (stimuli["PrimeLanguage"]  == LANGUAGE_NAME[prime_lang] + "_Prime") &
        (stimuli["TargetLanguage"] == LANGUAGE_NAME[target_lang] + "_Target")
    ]
    if verb_condition is not None:
        rows = rows[rows["ExtraCondition"] == verb_condition]

    results = []
    # One item = one Set in one verb condition; it has all four prime x target rows.
    for (set_id, condition), item in rows.groupby(["Set", "ExtraCondition"]):
        # Score every (prime structure, target structure) combination once.
        logp = {}
        for _, r in item.iterrows():
            prime_structure  = structure_of(r["PrimeType"])
            target_structure = structure_of(r["TargetType"])
            logp[(prime_structure, target_structure)] = conditional_logprob(
                model, r["PrimeSentence"], r["TargetSentence"])

        if set(logp) != {("PO", "PO"), ("PO", "DO"), ("DO", "PO"), ("DO", "DO")}:
            continue  # skip any item missing one of the four combinations

        # For each target structure, compare congruent vs. incongruent prime.
        for target_structure, other in (("PO", "DO"), ("DO", "PO")):
            prob_match = relative_probability(  # congruent prime = same structure
                logp[(target_structure, target_structure)], logp[(target_structure, other)])
            prob_mismatch = relative_probability(  # incongruent prime = other structure
                logp[(other, target_structure)], logp[(other, other)])
            results.append({"Set": set_id, "verb_condition": condition,
                            "target_structure": target_structure,
                            "prob_match": prob_match, "prob_mismatch": prob_mismatch,
                            "PE": prob_match - prob_mismatch})

    return pd.DataFrame(results, columns=["Set", "verb_condition", "target_structure",
                                          "prob_match", "prob_mismatch", "PE"])

## 6. Does priming cross languages? (Dutch L1 -> English L2)

`model_nl_en` learned Dutch first, so following our convention we prime in **Dutch (L1)** and measure
the effect on **English (L2)** targets.

We plot the target's **relative probability** after a congruent vs. an incongruent prime, **separately
for each target structure**. For each structure, the congruent prime raising its relative probability above the incongruent
prime is priming. Note the two structures mirror each other around the dotted 0.5 line: the model has
an overall preference between PO and DO, so one structure is below 0.5 and the other above, but the
congruent-vs-incongruent **gap** (the priming effect) is the same for both.

In [ ]:
import matplotlib.pyplot as plt

# L1 -> L2: prime in Dutch (L1), target English (L2).
baseline = run_priming_experiment(model_nl_en, stimuli, prime_lang="nl", target_lang="en")

print("Mean priming effect (Dutch L1 -> English L2):", round(baseline["PE"].mean(), 3))

# Relative probability of each target structure after a congruent vs. incongruent prime.
# (Arnett shows one structure per panel; we show both so the mirroring around 0.5 is visible.)
by_structure = (baseline.groupby("target_structure")[["prob_match", "prob_mismatch"]]
                .mean().reindex(["PO", "DO"]))
by_structure.columns = ["congruent prime", "incongruent prime"]

ax = by_structure.plot(kind="bar", figsize=(5.5, 3.3), color=["#E69F00", "#9C6ADE"], rot=0)
ax.axhline(0.5, color="black", linewidth=0.8, linestyle=":")
ax.set_ylim(0, 1)
ax.set_xlabel("target structure")
ax.set_ylabel("target relative probability")
ax.set_title("Dutch prime (L1) -> English target (L2)")
ax.legend(title=None)
plt.tight_layout()
plt.show()

*A note on exact numbers.* This reproduces Arnett et al.'s **pattern** (congruent > incongruent,
with PO dispreferred); our bars sit a little lower than their figure - most probably because their surprisals are
computed on input wrapped in the special tokens the model was trained with (`[CLS] … [SEP]`), whereas
`minicons` scores the bare `"prime. target"` string.

## 7. Does it matter which language was learned first?

Following Arnett et al, we keep the **prime = L1, target = L2** convention fixed,
and instead change **which language was learned first** by loading the model with the opposite
exposure order:

- `model_en_nl` (English L1 → Dutch L2): prime English, target **Dutch**.
- `model_nl_en` (Dutch L1 → English L2): prime Dutch, target **English**.

Both are L1→L2 priming; the only difference is the exposure order, which here also flips the target
language. Crucially, a single model only gives you *one* L1→L2 direction - so to get both a
Dutch-target and an English-target measurement you **need both models**. That is exactly why Arnett
et al. trained reversed-exposure pairs: it is the only way to separate "which language is the target"
from "which language is L1 vs L2".

In [ ]:
model_en_nl = load_scorer("catherinearnett/B-GPT_en_nl_simultaneous")  # English L1, Dutch L2

# Each model primes in its OWN L1 and targets its OWN L2 (Arnett's convention).
dutch_first   = run_priming_experiment(model_nl_en, stimuli, prime_lang="nl", target_lang="en")  # target English
english_first = run_priming_experiment(model_en_nl, stimuli, prime_lang="en", target_lang="nl")  # target Dutch

effects = pd.Series({
    "Dutch first\n(target: English)": dutch_first["PE"].mean(),
    "English first\n(target: Dutch)": english_first["PE"].mean(),
})
print("Priming effect (congruent - incongruent relative probability), L1 -> L2:")
print(effects.round(3))

ax = effects.plot(kind="bar", figsize=(5, 3.2), color="#4C72B0", rot=0)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("priming effect")
plt.title("Is the priming effect stronger when the target is English?")
plt.tight_layout()
plt.show()

*Optional aside - the L2→L1 direction.* Arnett et al. report L1→L2 in the main text and the
reverse only in an appendix. If you want to probe priming in the L2→L1 direction, swap prime and
target on a single model, e.g. `run_priming_experiment(model_nl_en, stimuli, "en", "nl")` (prime
English = L2, target Dutch = L1). Keep it clearly separate from the L1→L2 results above.

## 8. Does priming grow with more exposure? (with a control)

These models are released with saved **checkpoints** from many points in training. Each checkpoint
name is a training-step number (0 = start, the largest = fully trained). The **second language is
introduced halfway through training**, so to watch crosslingual priming appear we sample a few
checkpoints: one *before* the second language, several just *after* it is introduced (where the
effect should emerge), and a couple later on. Each checkpoint is downloaded separately, so this cell
takes a few minutes.

At each checkpoint we measure **two** things:

- **crosslingual priming** - Dutch prime (L1) → English target (L2), the effect of interest, and
- a **within-Dutch control** - Dutch prime → Dutch target (priming within a language the model has
  known from the very start). The prime language is Dutch in both, so the control isolates what
  changes when the *target* is the newly learned language.

Before English is introduced the model has never seen English, so it cannot prime an English target
and the crosslingual effect should sit **near zero**; the within-Dutch control should already be
high. The informative result is the crosslingual line **rising after English is introduced**, toward
the control, as the model acquires English grammar.

In [ ]:
from huggingface_hub import list_repo_refs

model_id = "catherinearnett/B-GPT_nl_en_simultaneous"
references = list_repo_refs(model_id, token=HF_TOKEN or None)
available = [ref.name for ref in references.branches] + [ref.name for ref in references.tags]

# Checkpoint names are training-step numbers. Sort them, and note that the second
# language is introduced halfway through training.
steps = sorted(int(name) for name in available if name.isdigit())
final_step = steps[-1]
second_language_onset = final_step // 2


def nearest_step(target):
    """Return the available checkpoint closest to `target` training steps."""
    return min(steps, key=lambda step: abs(step - target))


# Sample: one point before the second language, several just after it is introduced
# (where the effect should appear), and a couple later in training.
targets = [second_language_onset // 2,            # before the second language
           second_language_onset,                 # the moment it is introduced
           second_language_onset + 200,           # shortly after
           second_language_onset + 1000,          # a bit later
           (second_language_onset + final_step) // 2,   # midway through the rest
           final_step]                            # fully trained
checkpoints = sorted({nearest_step(t) for t in targets})

print(f"Second language introduced at step {second_language_onset} of {final_step}.")
print("Measuring priming at steps:", checkpoints)

In [ ]:
measurements = []
for step in checkpoints:
    model_at_step = load_scorer(model_id, revision=str(step))
    # One model load, two measurements: the crosslingual effect (L1->L2) and the control.
    crosslingual  = run_priming_experiment(model_at_step, stimuli, "nl", "en")["PE"].mean()
    within_dutch  = run_priming_experiment(model_at_step, stimuli, "nl", "nl")["PE"].mean()
    measurements.append({"step": step,
                         "crosslingual": crosslingual,
                         "within_dutch": within_dutch})
    print(f"step {step:>7}:  Dutch->English = {crosslingual:+.3f}   "
          f"Dutch->Dutch (control) = {within_dutch:+.3f}")

progress = pd.DataFrame(measurements).sort_values("step")
plt.figure(figsize=(6.5, 3.5))
plt.plot(progress["step"], progress["crosslingual"],
         "o-", label="Dutch -> English (crosslingual)")
plt.plot(progress["step"], progress["within_dutch"],
         "s--", color="gray", label="Dutch -> Dutch (control)")
plt.axvline(second_language_onset, color="black", linestyle=":",
            linewidth=1, label="second language introduced")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("training step")
plt.ylabel("mean priming effect")
plt.title("Crosslingual priming vs. within-Dutch control")
plt.legend()
plt.tight_layout()
plt.show()

**Reading the plot.** The grey control line is within-Dutch priming - Dutch is the L1, known
from the start, so it is high throughout and acts as a rough ceiling. The blue crosslingual line
(Dutch -> English) starts near zero: before English is introduced the model has no English, so it
cannot prime an English target. After the dotted line, as the model learns English, the crosslingual
effect is steadily increasing toward the control - the emergence of grammatical structure shared between the two
languages.

## 9. The lexical boost (same verb vs different verb)

In people, structural priming is *stronger* when the prime and the target share the same verb - the
"lexical boost". The stimuli mark whether each prime uses the same verb as its target, so we can
check whether the model shows the same pattern.

In [ ]:
available_conditions = set(stimuli["ExtraCondition"].unique())

for condition in ["same verb", "different verb"]:
    if condition not in available_conditions:
        print(f"{condition:>15}: not present in this stimulus set (skipped)")
        continue
    priming = run_priming_experiment(model_nl_en, stimuli, "nl", "en",
                                     verb_condition=condition)["PE"].mean()
    print(f"{condition:>15}: mean priming = {priming:+.3f}")

## 10. (Optional) More distant languages

The B-GPT models also include pairs less similar to English, such as Polish-English
(`B-GPT_pl_en_simultaneous`) and Greek-English (`B-GPT_el_en_simultaneous`). With the matching
Polish or Greek stimulus file (same columns, from the OSF page), you can test whether priming is
weaker between less similar languages. Keep the same L1→L2 convention with English as the target -
prime in the other language (L1), target English (L2) - so the only thing changing from the
Dutch-English baseline is how far the prime language is from English. The cell below is a starting
point.

In [ ]:
# model_pl_en = load_scorer("catherinearnett/B-GPT_pl_en_simultaneous")  # Polish L1, English L2
# stimuli_pl  = load_stimuli("priming_pl_en.tsv")   # same columns: Polish primes, English targets
# print(run_priming_experiment(model_pl_en, stimuli_pl, "pl", "en")["PE"].mean())  # L1 -> L2, English target
print("Upload a Polish or Greek stimulus file and compare its priming (-> English) to Dutch -> English.")

## 11. Things to think about

- You changed one thing at a time (the direction, the amount of training, the verb overlap). Which
  change made the biggest difference to priming?
- The model's "priming effect" is a change in probability. In people, priming is usually measured as
  a choice between structures when speaking. How good is the analogy?
- Priming tells us about the model's *behaviour*. What would you need to look at to say something
  about how the two languages are *represented inside* the model?